# Compare MEG and iEEG component spaces

This notebook displays results from `compare_subspace.py`; run the expensive analysis on the cluster first. The pipeline supports independent PCA, PLSSVD and joint PCA with identical trial splits and preprocessing.

**Primary scope:** new trials from the same participants and locations. Trial partitions are train/tune/test A/test B. Dimensions are a prespecified sensitivity grid, not selected on test similarity. Repetitions refit the entire model and vary trial splits and MEG pairing; their range is descriptive, not a confidence interval.

Temporal representations contain condition × time rows. Spatial representations are forward patterns evaluated on the common electrode grid. Models using full MEG data are fitted on all sources before sampling patterns at the mapped electrode locations. Repeated nearest-source mappings are retained and saved; spatial rows are not independent observations.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / 'compare_subspace.py').exists() and (ROOT / 'iEEGvsMEG' / 'compare_subspace.py').exists():
    ROOT = ROOT / 'iEEGvsMEG'
sys.path.insert(0, str(ROOT))
from compare_subspace import load_results, plot_results

MEG_KIND = 'paired_coverage'  # full_average, full_concatenated, coverage_average, paired_coverage, random_control
OUTPUT_DIR = ROOT / 'out' / 'compare_subspace' / MEG_KIND
DISPLAY_K = 3  # Must be one of the dimensions saved in the run.


## Run the analysis

The input is the **complete** `out/trial_cache` created by `plssvd_eval.py` / `prepare_trial_cache`. An incomplete cache is rejected. Use the same Python environment and project helpers as the PLSSVD workflow; this comparison also requires scikit-learn.

Run on the cluster:
```bash
python -u compare_subspace.py --root /path/to/iEEGvsMEG \
    --meg-kind paired_coverage --models separate_pca plssvd joint_pca \
    --dimensions 1 2 3 5 10 --repeats 5 --block-scaling equal_variance
```

Use `--output-dir` for a separate run/configuration. `equal_variance` balances total training variance across modalities; `none` preserves their relative preprocessed amplitudes. The mean and scale are always estimated on training trials. Group-dependent trials require `--split-unit group` and trial metadata in the cache.

The script saves all data and the four figures as PNG/PDF. Copy the output directory back here to inspect it. Set `RUN_ANALYSIS=True` below only to launch computation from this notebook.


In [ ]:
RUN_ANALYSIS = False
if RUN_ANALYSIS:
    import subprocess
    subprocess.run([
        sys.executable, '-u', str(ROOT / 'compare_subspace.py'),
        '--root', str(ROOT), '--meg-kind', MEG_KIND,
        '--models', 'separate_pca', 'plssvd', 'joint_pca',
        '--dimensions', '1', '2', '3', '5', '10', '--repeats', '5',
        '--output-dir', str(OUTPUT_DIR),
    ], check=True)


In [ ]:
if not (OUTPUT_DIR / 'config.json').exists():
    raise FileNotFoundError(f'Run compare_subspace.py on the cluster, then copy its results to {OUTPUT_DIR}')
config, tables = load_results(OUTPUT_DIR)
display(pd.Series(config, name='Run configuration'))
# Regenerate/export figures from small tables; no raw trial data are loaded.
plot_results(OUTPUT_DIR, k=DISPLAY_K, show=False)


## 1. Subspace overlap versus dimension

Mean squared cosine of principal angles between centered column spaces: 1 means identical subspaces at full requested rank; 0 means orthogonal. Rank loss is reported and missing dimensions count as zero. Compare methods at the same dimension; larger dimensions can increase overlap mechanically.

Temporal scores and spatial patterns answer distinct questions. High overlap alone does not demonstrate condition-specific biology. Plot lines show medians, and shading shows the range across overlapping repetitions.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'subspace_overlap.png')))
display(tables['overlap'].query("partition == 'test'")[
    ['model', 'repeat', 'space', 'k', 'overlap', 'rank_a', 'rank_b', 'angles_deg']])


## 2. Held-out alignment error by transformation complexity

Each modality is centered and divided by one scalar RMS estimated on training representations, preserving its relative axis geometry. Compare identity, orthogonal rotation/reflection, regularized affine, and regularized quadratic mappings in both directions. Ridge penalties are selected using **tuning** error independently for each model, dimension, direction and representation.

Normalized error = `||prediction − target|| / ||target − training target mean||`. Values below 1 beat the training-mean baseline; `Q² = 1 − error²`. Mappings are frozen before evaluating independent test trial averages. Spatial locations and time rows recur across partitions; this is not held-out-location or held-out-time generalization. Flexible transformations should earn their complexity through reproducible test improvements.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / f'alignment_complexity_k{DISPLAY_K}.png')))
display(tables['alignment'].query("partition == 'test' and k == @DISPLAY_K")[
    ['model', 'repeat', 'space', 'direction', 'complexity', 'alpha', 'nrmse', 'q2']])


## 3. Within-modality reliability

Compare test A and test B using the same trained projection axes. This measures reliability of new-trial scores and forward patterns conditional on the model; repetitions additionally refit the model, but the bands are not independent-refit reliability estimates or confidence intervals. The table includes signed correlations of corresponding fixed components.

Use within-modality reproducibility as context for cross-modal overlap, not as a calibrated numerical ceiling. Trials are split within each modality and participant; no trial-by-trial pairing across MEG and iEEG is assumed.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'within_modality_reliability.png')))
display(tables['reliability'].query('k == @DISPLAY_K')[
    ['model', 'repeat', 'space', 'modality', 'overlap', 'mean_signed_r', 'component_r']])


## 4. Cluster stability and agreement

K-means uses all retained dimensions of spatial patterns, separately for each modality. A common cluster count is selected by mean tuning silhouette, never by test agreement. Because orthogonal transformations preserve Euclidean distances, rotating before independent K-means adds no information; Procrustes and clustering are evaluated separately.

- **Cross-modal ARI:** agreement on the same ordered electrode locations using held-out patterns assigned to training centroids.
- **Fixed-centroid stability:** test A versus test B assignments to frozen training centroids, within a modality.
- **Refit stability:** independently cluster test A and test B using the training/tuning-selected cluster count, then compare partitions. Projection axes remain fixed.

ARI is invariant to cluster label numbering; 1 indicates identical partitions and 0 is chance-adjusted agreement. Silhouette is a compactness diagnostic, not evidence that discrete biological clusters exist. These cluster metrics are descriptive and do not treat correlated electrodes as independent inferential samples.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'cluster_stability_agreement.png')))
display(tables['clusters'].query('k == @DISPLAY_K'))
display(tables['cluster_selection'].query('k == @DISPLAY_K'))


## Saved representations and audit trail

`*_model.npz` contains modality weights, training feature means/scales and train/tune/test scores. `*_spatial_patterns.npz` contains forward patterns for each retained dimension and partition. Alignment files contain the fitted maps and training normalization; cluster files contain centroids and labels. CSV files record feature identity, electrode mapping, participant pairing, trial partitions and preprocessing parameters.

PCA/PLSSVD weight spaces and forward pattern spaces are distinct. Anatomical interpretation here uses patterns; scores and weights remain available for additional explicitly labeled analyses. No component sign or ordering is reselected on test data.


In [ ]:
# Example: inspect one saved model without loading raw trials.
import numpy as np
MODEL = config['models'][0]
with np.load(OUTPUT_DIR / f'{MODEL}_000_model.npz') as saved:
    display(pd.DataFrame([{'array': key, 'shape': saved[key].shape} for key in saved.files]))
